In [0]:
from pyspark.sql import functions as F

In [0]:
sales = spark.table(
    "workspace.supermarket.sales_transactions"
)

inventory = spark.table(
    "workspace.supermarket.inventory"
)

products = spark.table(
    "workspace.supermarket.products"
)

suppliers = spark.table(
    "workspace.supermarket.suppliers"
)

purchase_orders = spark.table(
    "workspace.supermarket.purchase_orders"
)

In [0]:
velocity = sales.groupBy(
    "store_id",
    "product_id"
).agg(
    F.sum("quantity_sold").alias(
        "total_units_sold"
    ),

    F.min("sale_date").alias(
        "first_sale_date"
    ),

    F.max("sale_date").alias(
        "last_sale_date"
    )
).withColumn(
    "selling_days",

    F.datediff(
        F.col("last_sale_date"),
        F.col("first_sale_date")
    ) + 1
).withColumn(
    "avg_daily_sales",

    F.round(
        F.col("total_units_sold") /
        F.col("selling_days"),
        2
    )
)

display(velocity)

In [0]:
supplier_lead_time = purchase_orders.filter(
    F.col("delivery_date").isNotNull()
).groupBy(
    "supplier_id"
).agg(
    F.round(
        F.avg(
            F.datediff(
                "delivery_date",
                "order_date"
            )
        ),
        2
    ).alias("avg_lead_time_days")
)

display(supplier_lead_time)

In [0]:
product_supplier = products.select(
    "product_id",
    "product_name",
    "category",
    "supplier_id"
).join(
    supplier_lead_time,
    on="supplier_id",
    how="left"
)

display(product_supplier)

In [0]:
reorder_df = velocity.join(
    inventory.select(
        "store_id",
        "product_id",
        "quantity_on_hand"
    ),
    on=["store_id", "product_id"],
    how="left"
).join(
    product_supplier,
    on="product_id",
    how="left"
)

display(reorder_df)

In [0]:
reorder_df = reorder_df.withColumn(
    "safety_stock",
    F.lit(20)
).withColumn(
    "reorder_point",

    F.round(
        F.col("avg_daily_sales") *
        F.col("avg_lead_time_days") +
        F.col("safety_stock"),
        2
    )
)

In [0]:
reorder_df = reorder_df.withColumn(
    "recommended_order_quantity",

    F.when(
        F.col("quantity_on_hand") <
        F.col("reorder_point"),

        F.ceil(
            F.col("reorder_point") -
            F.col("quantity_on_hand")
        )
    ).otherwise(0)
)

In [0]:
reorder_df = reorder_df.withColumn(
    "recommendation",

    F.when(
        F.col("quantity_on_hand") == 0,
        "URGENT REORDER"
    ).when(
        F.col("quantity_on_hand") <
        F.col("reorder_point"),
        "REORDER"
    ).otherwise(
        "NO ACTION"
    )
)

In [0]:
display(
    reorder_df.select(
        "store_id",
        "product_id",
        "product_name",
        "category",
        "quantity_on_hand",
        "avg_daily_sales",
        "avg_lead_time_days",
        "safety_stock",
        "reorder_point",
        "recommended_order_quantity",
        "recommendation"
    ).orderBy(
        F.desc("recommended_order_quantity")
    )
)

In [0]:
reorder_df.write \
    .format("delta") \
    .mode("overwrite") \
    .saveAsTable(
        "workspace.supermarket.reorder_recommendations"
    )

In [0]:
display(
    spark.table(
        "workspace.supermarket.reorder_recommendations"
    )
)

In [0]:
final_recommendations = reorder_df.select(
    "store_id",
    "product_id",
    "product_name",
    "category",
    "quantity_on_hand",
    F.round(
        "avg_daily_sales",
        2
    ).alias("avg_daily_sales"),
    F.round(
        "avg_lead_time_days",
        2
    ).alias("lead_time_days"),
    "safety_stock",
    F.round(
        "reorder_point",
        2
    ).alias("reorder_point"),
    "recommended_order_quantity",
    "recommendation"
)

In [0]:
display(
    final_recommendations.orderBy(
        F.desc("recommended_order_quantity")
    )
)

In [0]:
final_recommendations.write \
    .mode("overwrite") \
    .option("header", True) \
    .csv(
        "/Volumes/workspace/supermarket/raw_data/reorder_recommendations"
    )

In [0]:
reorder_df = reorder_df.withColumn(
    "stockout_risk",

    F.when(
        F.col("quantity_on_hand") == 0,
        "HIGH"
    ).when(
        F.col("quantity_on_hand") <
        F.col("reorder_point"),
        "MEDIUM"
    ).otherwise(
        "LOW"
    )
)

In [0]:
display(
    reorder_df.select(
        "store_id",
        "product_name",
        "quantity_on_hand",
        "reorder_point",
        "stockout_risk",
        "recommendation"
    ).orderBy(
        F.desc("stockout_risk"),
        F.desc("recommended_order_quantity")
    )
)